# 💳 Credit Card Fraud Detection — Full Training Pipeline

**M.Tech Project | Dataset: ULB Credit Card Fraud Detection (Kaggle)**

### Pipeline Overview
1. Exploratory Data Analysis (EDA)
2. Chronological 60/20/20 train-val-test split
3. Preprocessing (no leakage)
4. Model training: Logistic Regression, Random Forest, XGBoost
5. Hyperparameter tuning with RandomizedSearchCV + TimeSeriesSplit
6. Optional SMOTE experiment
7. Threshold optimization on validation set
8. Final evaluation on test set (run ONCE)
9. Save all artifacts

## 0. Imports & Global Config

In [ ]:
import warnings
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ── Global config ─────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Search dynamically for creditcard.csv in /kaggle/input
import glob
matches = glob.glob('/kaggle/input/**/creditcard.csv', recursive=True)
if matches:
    DATASET_PATH = matches[0]
    print(f'Found dataset at: {DATASET_PATH}')
else:
    print('Contents of /kaggle/input:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'No input dir')
    DATASET_PATH = '/kaggle/input/creditcardfraud/creditcard.csv'
OUTPUT_DIR   = '/kaggle/working/'

FEATURE_COLS = [f'V{i}' for i in range(1, 29)] + ['Amount']
TARGET_COL   = 'Class'

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('All imports successful.')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

## 1. Exploratory Data Analysis (EDA)

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values ✓')

print(f'\n=== Duplicates ===')
n_dup = df.duplicated().sum()
print(f'{n_dup:,} duplicate rows found')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')

print(f'\n=== Class Distribution ===')
class_counts = df[TARGET_COL].value_counts()
fraud_pct = class_counts[1] / len(df) * 100
print(class_counts)
print(f'Fraud rate: {fraud_pct:.4f}%  →  heavily imbalanced!')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EDA — Credit Card Fraud Dataset', fontsize=15, fontweight='bold')

# 1. Class imbalance bar chart
ax = axes[0, 0]
class_counts.plot(kind='bar', ax=ax, color=['steelblue', 'crimson'], edgecolor='black')
ax.set_title('Class Distribution')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

# 2. Transaction Amount distribution (log scale)
ax = axes[0, 1]
df[df[TARGET_COL]==0]['Amount'].hist(ax=ax, bins=60, alpha=0.6, color='steelblue', label='Legit', density=True)
df[df[TARGET_COL]==1]['Amount'].hist(ax=ax, bins=60, alpha=0.6, color='crimson',   label='Fraud', density=True)
ax.set_title('Transaction Amount Distribution')
ax.set_xlabel('Amount')
ax.set_ylabel('Density')
ax.set_yscale('log')
ax.legend()

# 3. Time distribution
ax = axes[1, 0]
df[df[TARGET_COL]==0]['Time'].hist(ax=ax, bins=60, alpha=0.6, color='steelblue', label='Legit', density=True)
df[df[TARGET_COL]==1]['Time'].hist(ax=ax, bins=60, alpha=0.6, color='crimson',   label='Fraud', density=True)
ax.set_title('Time (seconds elapsed) Distribution')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Density')
ax.legend()

# 4. Amount box plot by class
ax = axes[1, 1]
df.boxplot(column='Amount', by=TARGET_COL, ax=ax,
           patch_artist=True,
           boxprops=dict(facecolor='lightblue'),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Amount by Class')
ax.set_xlabel('Class (0=Legit, 1=Fraud)')
ax.set_ylabel('Amount')
plt.sca(ax); plt.title('Amount by Class')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_overview.png'), bbox_inches='tight')
plt.show()
print('EDA overview saved.')

In [ ]:
# Top features most correlated with fraud
correlations = df.corr()[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
top_features = correlations.head(10).index.tolist()

print('Top 10 features correlated with Class (|r|):')
print(correlations.head(10).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
correlations.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Feature Correlation with Fraud (|r|)')
ax.set_xlabel('|Pearson correlation|')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_correlations.png'), bbox_inches='tight')
plt.show()

## 2. Chronological Train / Validation / Test Split (60 / 20 / 20)

> **No shuffling.** We sort by `Time` and cut at 60% and 80% marks. This mirrors real-world deployment where you train on past data and predict future fraud.

In [ ]:
df_sorted = df.sort_values('Time').reset_index(drop=True)
n = len(df_sorted)

train_end = int(n * 0.60)
val_end   = int(n * 0.80)

train_df = df_sorted.iloc[:train_end].copy()
val_df   = df_sorted.iloc[train_end:val_end].copy()
test_df  = df_sorted.iloc[val_end:].copy()

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_fraud = split[TARGET_COL].sum()
    pct     = n_fraud / len(split) * 100
    print(f'{name:5s}: {len(split):,} rows | Fraud: {n_fraud:,} ({pct:.3f}%)')

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_val   = val_df[FEATURE_COLS]
y_val   = val_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(f'\nscale_pos_weight (for XGBoost):')
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
SCALE_POS_WEIGHT = n_neg / n_pos
print(f'  neg={n_neg:,} / pos={n_pos:,} → ratio={SCALE_POS_WEIGHT:.2f}')

## 3. Preprocessing

> `StandardScaler` is **fit on training data only** and then applied to val/test. This prevents data leakage.

In [ ]:
def build_pipeline(classifier):
    """Wrap a classifier with a StandardScaler in a sklearn Pipeline."""
    return Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('clf',    classifier)
    ])

# Quick sanity check — fit scaler on train, transform val
_scaler_check = StandardScaler()
_scaler_check.fit(X_train)
print(f'Scaler fit on training data. Mean of Amount (train): {_scaler_check.mean_[-1]:.4f}')
print('Preprocessing pipeline helper ready.')

## 4. GPU Check for XGBoost

In [ ]:
def get_xgb_device():
    """Return 'cuda' if a GPU is available, else 'cpu'."""
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print('GPU detected — XGBoost will use CUDA.')
            return 'cuda'
    except FileNotFoundError:
        pass
    print('No GPU detected — XGBoost will use CPU.')
    return 'cpu'

XGB_DEVICE = get_xgb_device()

## 5. Define Models & Hyperparameter Grids

In [ ]:
# ── 5a. Base models ───────────────────────────────────────────────────────────
lr_base = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_base = RandomForestClassifier(
    class_weight='balanced_subsample',
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_base = XGBClassifier(
    scale_pos_weight=SCALE_POS_WEIGHT,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='aucpr',
    device=XGB_DEVICE,
    random_state=RANDOM_STATE,
    verbosity=0
)

# ── 5b. Hyperparameter search spaces ─────────────────────────────────────────
lr_param_dist = {
    'clf__C':      [0.001, 0.01, 0.1, 1, 10],
    'clf__solver': ['lbfgs', 'liblinear'],
}

rf_param_dist = {
    'clf__n_estimators':    [100, 200, 300],
    'clf__max_depth':       [None, 5, 10, 15],
    'clf__min_samples_leaf':[1, 2, 4],
    'clf__max_features':    ['sqrt', 'log2'],
}

xgb_param_dist = {
    'clf__learning_rate':    [0.01, 0.05, 0.1],
    'clf__max_depth':        [3, 5, 7],
    'clf__subsample':        [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0],
    'clf__n_estimators':     [200, 350, 500],
}

print('Models and search spaces defined.')

## 6. Hyperparameter Tuning with RandomizedSearchCV

> We use `TimeSeriesSplit(n_splits=3)` to respect temporal ordering during cross-validation. Scoring = `average_precision` (PR-AUC).

In [ ]:
# Combine train + val for tuning (with temporal CV, val is always in the future)
X_tune = pd.concat([X_train, X_val], ignore_index=True)
y_tune = pd.concat([y_train, y_val], ignore_index=True)

tscv = TimeSeriesSplit(n_splits=3)

def tune_model(name, base_clf, param_dist, n_iter=10):
    print(f'\n--- Tuning {name} (n_iter={n_iter}) ---')
    pipeline   = build_pipeline(base_clf)
    search     = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='average_precision',
        cv=tscv,
        refit=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_tune, y_tune)
    print(f'Best CV PR-AUC: {search.best_score_:.4f}')
    print(f'Best params   : {search.best_params_}')
    return search.best_estimator_

print('Tuning helper ready.')

In [ ]:
best_lr = tune_model('Logistic Regression', lr_base, lr_param_dist, n_iter=10)

In [ ]:
best_rf = tune_model('Random Forest', rf_base, rf_param_dist, n_iter=10)

In [ ]:
best_xgb = tune_model('XGBoost', xgb_base, xgb_param_dist, n_iter=10)

## 7. Optional Experiment — SMOTE

> SMOTE is applied **only to training data** after the split. We compare LR with and without SMOTE on the validation set. This is an experiment — the main pipeline does NOT use SMOTE.

In [ ]:
print('=== SMOTE Experiment (Logistic Regression only) ===')

# Scale train data first (fit on train only!)
scaler_smote = StandardScaler()
X_train_scaled = scaler_smote.fit_transform(X_train)
X_val_scaled   = scaler_smote.transform(X_val)

# Apply SMOTE only to training data
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE — Fraud: {y_train_smote.sum():,} | Legit: {(y_train_smote==0).sum():,}')

# Train LR on SMOTE data
lr_smote = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1
)
lr_smote.fit(X_train_smote, y_train_smote)

# Compare on validation set
proba_no_smote = best_lr.predict_proba(X_val)[:, 1]
proba_smote    = lr_smote.predict_proba(X_val_scaled)[:, 1]

pr_auc_no_smote = average_precision_score(y_val, proba_no_smote)
pr_auc_smote    = average_precision_score(y_val, proba_smote)

print(f'\nLR (no SMOTE) val PR-AUC : {pr_auc_no_smote:.4f}')
print(f'LR (SMOTE)    val PR-AUC : {pr_auc_smote:.4f}')

winner = 'SMOTE' if pr_auc_smote > pr_auc_no_smote else 'No SMOTE'
print(f'→ Winner for LR          : {winner}')
print('\nNote: SMOTE experiment is for comparison only. Main pipeline uses class_weight.')

## 8. Threshold Optimization on Validation Set

> The default threshold of 0.5 is arbitrary. We find the threshold that maximises F1-score on the **validation set** for each model.

In [ ]:
def find_best_threshold(y_true, proba):
    """Return the threshold that maximises F1 on the given set."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # thresholds has one fewer element than precisions/recalls
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (
        precisions[:-1] + recalls[:-1] + 1e-9
    )
    best_idx   = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_f1     = f1_scores[best_idx]
    return best_thresh, best_f1

models = {
    'Logistic Regression': best_lr,
    'Random Forest':       best_rf,
    'XGBoost':             best_xgb,
}

val_probas    = {}
val_thresholds = {}

for name, model in models.items():
    proba = model.predict_proba(X_val)[:, 1]
    val_probas[name] = proba
    thresh, f1 = find_best_threshold(y_val, proba)
    val_thresholds[name] = thresh
    print(f'{name:22s} → best threshold: {thresh:.4f}  (val F1: {f1:.4f})')

## 9. Model Comparison on Validation Set

In [ ]:
records = []
for name, model in models.items():
    proba  = val_probas[name]
    thresh = val_thresholds[name]
    preds  = (proba >= thresh).astype(int)

    records.append({
        'Model':       name,
        'PR-AUC':      round(average_precision_score(y_val, proba), 4),
        'ROC-AUC':     round(roc_auc_score(y_val, proba), 4),
        'Precision':   round(precision_score(y_val, preds, zero_division=0), 4),
        'Recall':      round(recall_score(y_val, preds), 4),
        'F1':          round(f1_score(y_val, preds), 4),
        'Threshold':   round(thresh, 4),
    })

comparison_df = pd.DataFrame(records).sort_values('PR-AUC', ascending=False)
comparison_df = comparison_df.reset_index(drop=True)
comparison_df.index += 1
print('=== Validation Set — Model Comparison ===')
print(comparison_df.to_string())

comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=True)
print('\nSaved model_comparison.csv')

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model      = models[best_model_name]
best_threshold  = val_thresholds[best_model_name]
print(f'\n✅ Best model (by PR-AUC): {best_model_name}  |  threshold: {best_threshold:.4f}')

## 10. Precision-Recall Curves (Validation Set)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue', 'forestgreen', 'crimson']

for (name, model), color in zip(models.items(), colors):
    proba     = val_probas[name]
    pr_auc    = average_precision_score(y_val, proba)
    prec, rec, _ = precision_recall_curve(y_val, proba)
    ax.plot(rec, prec, label=f'{name} (PR-AUC={pr_auc:.3f})', color=color, lw=2)

baseline = y_val.mean()
ax.axhline(baseline, color='grey', linestyle='--', label=f'Baseline (random, {baseline:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Validation Set', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pr_curves.png'), bbox_inches='tight')
plt.show()
print('PR curves saved.')

## 11. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance / Coefficients', fontsize=13, fontweight='bold')

# Random Forest importances
rf_clf   = best_rf.named_steps['clf']
rf_imps  = pd.Series(rf_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
rf_imps.head(15).plot(kind='barh', ax=axes[0], color='forestgreen', edgecolor='black')
axes[0].set_title('Random Forest — Top 15 Features')
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance')

# XGBoost feature importances
xgb_clf  = best_xgb.named_steps['clf']
xgb_imps = pd.Series(xgb_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
xgb_imps.head(15).plot(kind='barh', ax=axes[1], color='crimson', edgecolor='black')
axes[1].set_title('XGBoost — Top 15 Features')
axes[1].invert_yaxis()
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), bbox_inches='tight')
plt.show()
print('Feature importance plot saved.')

## 12. Final Test Set Evaluation ⚠️

> **This cell runs only once.** The test set is untouched until now. Results here reflect real-world performance.

In [ ]:
print(f'Evaluating best model on TEST set: {best_model_name}')
print(f'Using optimised threshold         : {best_threshold:.4f}')
print('─' * 55)

test_proba = best_model.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= best_threshold).astype(int)

test_pr_auc  = average_precision_score(y_test, test_proba)
test_roc_auc = roc_auc_score(y_test, test_proba)
test_f1      = f1_score(y_test, test_preds)
test_prec    = precision_score(y_test, test_preds, zero_division=0)
test_recall  = recall_score(y_test, test_preds)

print(f'PR-AUC    : {test_pr_auc:.4f}   ← primary metric')
print(f'ROC-AUC   : {test_roc_auc:.4f}')
print(f'Precision : {test_prec:.4f}')
print(f'Recall    : {test_recall:.4f}')
print(f'F1        : {test_f1:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, test_preds, target_names=['Legitimate', 'Fraud']))

In [ ]:
cm = confusion_matrix(y_test, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix (counts)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Legit', 'Predicted Fraud'],
    yticklabels=['Actual Legit',    'Actual Fraud'],
    ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix — {best_model_name}\n(Test Set)', fontweight='bold')

# PR curve on test set
prec_t, rec_t, _ = precision_recall_curve(y_test, test_proba)
axes[1].plot(rec_t, prec_t, color='crimson', lw=2,
             label=f'PR-AUC = {test_pr_auc:.3f}')
axes[1].axhline(y_test.mean(), color='grey', linestyle='--', label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR Curve — {best_model_name} (Test Set)', fontweight='bold')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

## 13. Save All Artifacts

In [ ]:
# 1. Save best pipeline
pipeline_path = os.path.join(OUTPUT_DIR, 'best_pipeline.pkl')
joblib.dump(best_model, pipeline_path)
print(f'Saved pipeline   → {pipeline_path}')

# 2. Save threshold
threshold_path = os.path.join(OUTPUT_DIR, 'threshold.json')
with open(threshold_path, 'w') as f:
    json.dump({'threshold': float(best_threshold), 'model': best_model_name}, f, indent=2)
print(f'Saved threshold  → {threshold_path}')

# 3. Save evaluation results
eval_results = {
    'model':           best_model_name,
    'threshold':       float(best_threshold),
    'test': {
        'pr_auc':    round(float(test_pr_auc), 4),
        'roc_auc':   round(float(test_roc_auc), 4),
        'precision': round(float(test_prec), 4),
        'recall':    round(float(test_recall), 4),
        'f1':        round(float(test_f1), 4),
    },
    'validation_comparison': comparison_df.to_dict(orient='records'),
}
eval_path = os.path.join(OUTPUT_DIR, 'evaluation_results.json')
with open(eval_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'Saved evaluation → {eval_path}')

print('\n=== All artifacts saved to /kaggle/working/ ===')
print('Download them from the OUTPUT panel on the right.')

## 14. Summary

In [ ]:
print('=' * 60)
print(' CREDIT CARD FRAUD DETECTION — TRAINING COMPLETE')
print('=' * 60)
print(f'  Best Model     : {best_model_name}')
print(f'  Threshold      : {best_threshold:.4f}')
print(f'  Test PR-AUC    : {test_pr_auc:.4f}   ← primary metric')
print(f'  Test ROC-AUC   : {test_roc_auc:.4f}')
print(f'  Test F1        : {test_f1:.4f}')
print(f'  Test Precision : {test_prec:.4f}')
print(f'  Test Recall    : {test_recall:.4f}')
print('=' * 60)
print('\nArtifacts saved:')
for fname in ['best_pipeline.pkl', 'threshold.json', 'evaluation_results.json',
              'model_comparison.csv', 'pr_curves.png', 'confusion_matrix.png',
              'feature_importance.png', 'eda_overview.png']:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = '✓' if os.path.exists(fpath) else '✗'
    print(f'  {exists}  {fname}')